# Ablation_HDFS_FamilyB — train / eval on the 20260818 notebook graph

Self-contained **Family B** notebook. It trains AttributeAwareGAE on the same frozen HDFS bundle used by [`legacy/6_GAE_Training_Colab.ipynb`](../legacy/6_GAE_Training_Colab.ipynb):

`data/processed/hdfs/20260818_0002_1_parser_3_graph_dataset.pt.gz`

No Drain, no Deepseek, no graph rebuild. Model alterations are the BGL Family B set from `configs/ablation_train.yaml` / `configs/ablation_hdfs_train.yaml` (GINE agg, node transform, latent/hidden size, lr, loss weights, inner-product decoder, lexical node recon). Architecture defaults match that matrix: clean-train, GINE-sum, concat fusion, α=β=γ=1, 25 epochs, Adam lr **0.001** (BGL default, not `ablation_base.yaml`'s HDFS 0.01).

This is the **transductive 20260818 notebook graph** (~575k blocks, n_train≈390755). Do **not** mix it with `hdfs-full-monty-v1` or inductive Family A campaigns, and do not compare PR-AUC across those graphs.

On Colab, put the `.pt.gz` on Drive under `hybrid-log-analyzer-artifacts/data/processed/hdfs/`. Each finished seed (metrics + confusion matrix, learning curve, PR curve, score distribution with threshold) is copied to Drive immediately. Rank by **test PR-AUC** vs `baseline_full` (chance ≈ 0.03). `SMOKE=True` is 1 epoch and ≤5000 graphs — do not publish those metrics.


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")


In [ ]:
from pathlib import Path
import gzip
import os
import shutil
import sys

CAMPAIGN_ID = "hdfs-family-b-legacy-v1"
DATASET = "hdfs"
RUN_TRAIN = True
SMOKE = True  # pipeline check; set False for the published 13×5 GAE runs
SEEDS = [13, 29, 42, 71, 101]
SMOKE_GRAPH_CAP = 5000
REUSE_LOCAL_GRAPH_FILE = True
GRAPH_DATASET_RELATIVE_PATH = (
    "data/processed/hdfs/20260818_0002_1_parser_3_graph_dataset.pt.gz"
)
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")

ARM_OVERRIDES = {
    "baseline_full": {},
    "gine_mean_agg": {"gine_aggregation": "mean"},
    "gine_max_agg": {"gine_aggregation": "max"},
    "linear_node_transform": {"node_transformation": "linear"},
    "latent_32": {"latent_dim": 32},
    "hidden_256": {"hidden_dim": 256},
    "learning_rate_001": {"learning_rate": 0.001},
    "gamma_05": {"gamma": 0.5},
    "inner_product_structure": {"structure_decoder": "inner_product"},
    "alpha_0": {"alpha": 0.0},
    "beta_0": {"beta": 0.0},
    "gamma_0": {"gamma": 0.0},
    "lexical_node_recon": {"node_reconstruct": "without_sbert"},
}
EXPECTED_ARMS = tuple(ARM_OVERRIDES)


def find_local_workspace() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "notebooks").exists() and (candidate / "run_ablation.py").exists():
            return candidate
        marker = candidate / GRAPH_DATASET_RELATIVE_PATH
        if marker.exists() or marker.with_suffix("").exists():
            return candidate
    return here


WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else find_local_workspace()
GRAPH_DATASET_PATH = WORKSPACE_ROOT / GRAPH_DATASET_RELATIVE_PATH


def _copy_tree(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_file():
        shutil.copy2(source, destination)
        return
    shutil.copytree(source, destination, dirs_exist_ok=True)


def _copy_file_if_needed(source: Path, dest: Path) -> bool:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == source.stat().st_size and dest.stat().st_mtime >= source.stat().st_mtime:
        return False
    shutil.copy2(source, dest)
    return True


def push_to_drive(local: Path) -> Path | None:
    """Copy a workspace file or directory onto Drive. No-op off Colab."""
    local = Path(local)
    if not IN_COLAB or not local.exists():
        return None
    try:
        relative = local.resolve().relative_to(Path(WORKSPACE_ROOT).resolve())
    except ValueError:
        print(f"[DRIVE] skip (outside workspace): {local}")
        return None
    dest = DRIVE_ARTIFACT_ROOT / relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    copied = 0
    if local.is_file():
        copied += int(_copy_file_if_needed(local, dest))
    else:
        dest.mkdir(parents=True, exist_ok=True)
        for path in local.rglob("*"):
            if path.is_file():
                copied += int(_copy_file_if_needed(path, dest / path.relative_to(local)))
    print(f"[DRIVE] saved {relative} → {dest} ({copied} file(s) copied)")
    return dest


def push_run_outputs(output_dir: Path) -> None:
    push_to_drive(Path(output_dir))


def stage_legacy_graph() -> Path:
    """Copy the 20260818 HDFS bundle from Drive (Colab) or the checkout, then gunzip."""
    relative = Path(GRAPH_DATASET_RELATIVE_PATH)
    dest_gz = Path(WORKSPACE_ROOT) / relative
    dest_gz.parent.mkdir(parents=True, exist_ok=True)
    candidates = []
    if IN_COLAB:
        candidates.append(DRIVE_ARTIFACT_ROOT / relative)
    repo = find_local_workspace()
    candidates.extend(
        [
            repo / relative,
            Path.cwd() / relative,
        ]
    )
    source = next((path for path in candidates if path.exists()), None)
    if source is None and not dest_gz.exists():
        raise FileNotFoundError(
            f"Missing {relative}. Upload it to Drive "
            f"{DRIVE_ARTIFACT_ROOT / relative} or place it under the checkout."
        )
    if source is not None and source.resolve() != dest_gz.resolve():
        if not dest_gz.exists() or not REUSE_LOCAL_GRAPH_FILE:
            print(f"[GRAPH] copying {source} → {dest_gz}")
            shutil.copy2(source, dest_gz)
        else:
            print(f"[GRAPH] reusing local {dest_gz}")
    dest_pt = dest_gz.with_suffix("") if dest_gz.suffix == ".gz" else dest_gz
    if dest_gz.suffix == ".gz" and (not dest_pt.exists() or not REUSE_LOCAL_GRAPH_FILE):
        print(f"[GRAPH] decompressing {dest_gz.name} → {dest_pt.name}")
        with gzip.open(dest_gz, "rb") as incoming, dest_pt.open("wb") as outgoing:
            shutil.copyfileobj(incoming, outgoing, length=16 * 1024 * 1024)
    return dest_pt


print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"GRAPH          = {GRAPH_DATASET_RELATIVE_PATH}")
print(f"CAMPAIGN_ID    = {CAMPAIGN_ID}")
print(f"RUN_TRAIN={RUN_TRAIN}  SMOKE={SMOKE}  SEEDS={SEEDS}")
print(f"Family B arms  = {list(EXPECTED_ARMS)}")


## Colab environment

Keep Colab's CUDA PyTorch. Install PyG from the matching `data.pyg.org` wheel index. Stage the 20260818 graph from Drive into `/content/workspace` — do not train from the mount.


In [ ]:
def pyg_wheel_url() -> str:
    import torch

    torch_version = torch.__version__.split("+", maxsplit=1)[0]
    cuda_version = torch.version.cuda
    platform = f"cu{str(cuda_version).replace('.', '')}" if cuda_version else "cpu"
    return f"https://data.pyg.org/whl/torch-{torch_version}+{platform}.html"


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

    import torch

    print(f"Colab PyTorch {torch.__version__}  cuda={torch.cuda.is_available()}")
    wheel_url = pyg_wheel_url()
    print(f"PyG wheel index: {wheel_url}")
    get_ipython().run_line_magic(
        "pip",
        "install -q pandas scikit-learn tqdm pyyaml matplotlib",
    )
    get_ipython().run_line_magic("pip", f"install -q torch-geometric -f {wheel_url}")
    try:
        get_ipython().run_line_magic(
            "pip",
            f"install -q --only-binary=:all: pyg_lib torch_scatter torch_sparse -f {wheel_url}",
        )
    except Exception as exc:
        print(f"Optional PyG extension wheels unavailable ({exc}); using PyTorch fallbacks.")

    import torch_geometric

    print(f"torch-geometric {torch_geometric.__version__}")
    drive_outputs = DRIVE_ARTIFACT_ROOT / "outputs" / DATASET
    local_outputs = WORKSPACE_ROOT / "outputs" / DATASET
    if drive_outputs.exists():
        print(f"[DRIVE] staging prior outputs {drive_outputs} → {local_outputs}")
        _copy_tree(drive_outputs, local_outputs)
else:
    import torch
    import torch_geometric

    mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    print(f"Local PyTorch {torch.__version__}  pyg={torch_geometric.__version__}")
    print(f"cuda={torch.cuda.is_available()}  mps={mps}")

GRAPH_DATASET_PATH = stage_legacy_graph()
print(f"Prepared graph dataset: {GRAPH_DATASET_PATH}")


## Inspect frozen graph

Confirm the bundle matches the legacy Colab trainer (all-block HDFS, existing train/val/test indices).


In [ ]:
import torch

bundle = torch.load(GRAPH_DATASET_PATH, map_location="cpu", weights_only=False)
required = {"data_list", "idx_train", "idx_val", "idx_test", "node_dim", "edge_dim"}
missing = required - set(bundle)
if missing:
    raise ValueError(f"Graph dataset is missing: {sorted(missing)}")
n_train = len(bundle["idx_train"])
n_val = len(bundle["idx_val"])
n_test = len(bundle["idx_test"])
print(f"Graphs: {len(bundle['data_list']):,}")
print(f"Node features: {bundle['node_dim']}; edge features: {bundle['edge_dim']}")
print(f"Saved splits — train: {n_train:,}, validation: {n_val:,}, test: {n_test:,}")
meta = bundle.get("dataset_meta") or {}
print(f"dataset_meta keys: {sorted(meta)[:20]}")
identity = meta.get("graph_identity")
if identity:
    print(f"graph_identity: {identity}")
del bundle


## Inlined AttributeAwareGAE + train / eval / report

Same clean-train loop as the HDFS Full Monty notebook (val-F1 checkpoint, frozen test threshold). Only `ARM_OVERRIDES` change. Outputs go to `outputs/hdfs/{campaign}_{arm}_seed{seed}/` and are copied to Drive after each seed, including `figures/confusion_matrix.png`, `learning_curve.png`, `test_pr_curve.png`, and `test_score_distribution.png` (threshold marked).


In [ ]:
"""Attribute-Aware Graph Autoencoder (AttributeAwareGAE).

Architecture extracted from ``6_GAE_Training_BGL_fixed.ipynb`` and corrected
so structure reconstruction matches a *directed*, *per-graph* adjacency:

Encoder
    * ``raw_node_norm`` — BatchNorm1d on input node features.
    * ``node_proj`` + ``edge_proj`` — linear projections to ``hidden_dim``.
    * ``encoder_conv`` — GINEConv with configurable aggregation.

Decoder (multi-task)
    1. Structure — directed concat-MLP (default) or inner product. Trained
       with BCE on observed edges and *in-graph* non-edges (never cross-graph
       pairs from a PyG mini-batch).
    2. Node feature reconstruction — 2-layer MLP from latent Z onto the
       reconstruction columns (full ``x`` or TF-IDF+extras when
       ``node_reconstruct=without_sbert``).
    3. Edge attribute reconstruction — 2-layer MLP from ⟨Z_i ∥ Z_j⟩.

``raw_edge_norm`` is intentionally absent: BGL's ``log1p(td_std)`` is ~0 for
most edges and in-model BatchNorm would explode. Edges are pre-normalised
on the training split with std clamped ≥ 0.1.
"""

from __future__ import annotations

_LOCK_REF = globals().get('_LOCK_REF')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv
from torch_geometric.utils import negative_sampling, scatter

FULL_STRUCTURE_MAX_NODES = 256
NODE_EXTRA_DIM = 9


def node_recon_index_tensor(
    node_dim: int,
    *,
    sbert_dim: int,
    extra_dim: int = NODE_EXTRA_DIM,
) -> torch.Tensor:
    """Column indices for the node decoder when skipping the SBERT block."""
    if sbert_dim <= 0:
        return torch.arange(node_dim, dtype=torch.long)
    embed_dim = node_dim - extra_dim
    if embed_dim < sbert_dim:
        raise ValueError(
            f"sbert_dim={sbert_dim} exceeds embedding width {embed_dim} "
            f"(node_dim={node_dim}, extra_dim={extra_dim})."
        )
    tfidf_dim = embed_dim - sbert_dim
    lexical = torch.arange(tfidf_dim, dtype=torch.long)
    extras = torch.arange(embed_dim, node_dim, dtype=torch.long)
    return torch.cat([lexical, extras])


class AttributeAwareGAE(nn.Module):
    """Multi-task Graph Autoencoder with a GINEConv encoder.

    Parameters
    ----------
    structure_decoder : str
        ``"mlp"`` (directed concat-MLP, default) or ``"inner_product"``
        (symmetric ⟨z_i, z_j⟩, Family B ablation).
    """

    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 128,
        latent_dim: int = 64,
        gine_aggregation: str = "sum",
        node_transformation: str = "mlp",
        structure_decoder: str = "mlp",
        node_recon_index: torch.Tensor | None = None,
        tfidf_dim: int = 0,
        sbert_dim: int = 0,
        fusion_mode: str = "concat",
        modality_projection_dim: int = 64,
        node_loss_mode: str = "global",
        node_block_weights: dict[str, float] | None = None,
    ) -> None:
        super().__init__()
        if structure_decoder not in {"mlp", "inner_product"}:
            raise ValueError(
                f"structure_decoder must be 'mlp' or 'inner_product', got {structure_decoder!r}"
            )
        self.structure_decoder_kind = structure_decoder
        self.latent_dim = latent_dim
        self.node_dim = int(node_dim)
        self.tfidf_dim = int(tfidf_dim)
        self.sbert_dim = int(sbert_dim)
        self.extra_dim = int(node_dim - self.tfidf_dim - self.sbert_dim)
        if self.extra_dim < 0:
            raise ValueError("tfidf_dim + sbert_dim cannot exceed node_dim.")
        if fusion_mode not in {"concat", "projected_gated"}:
            raise ValueError("fusion_mode must be 'concat' or 'projected_gated'.")
        if node_loss_mode not in {"global", "block_balanced"}:
            raise ValueError("node_loss_mode must be 'global' or 'block_balanced'.")
        self.fusion_mode = fusion_mode
        self.node_loss_mode = node_loss_mode
        self.node_block_weights = {
            "tfidf": 1.0,
            "sbert": 1.0,
            "extras": 1.0,
            **(node_block_weights or {}),
        }
        if node_recon_index is None:
            recon_index = torch.arange(node_dim, dtype=torch.long)
        else:
            recon_index = torch.as_tensor(node_recon_index, dtype=torch.long).reshape(-1)
        if recon_index.numel() == 0:
            raise ValueError("node_recon_index must contain at least one column.")
        # Not a state-dict buffer: old packages must keep the historical key set
        # when recon_dim == node_dim. The index is saved on the training checkpoint.
        self.node_recon_index = recon_index
        recon_dim = int(recon_index.numel())

        self.raw_node_norm = nn.BatchNorm1d(node_dim, affine=False)

        self.modality_projectors = nn.ModuleDict()
        self.modality_gate_logits = None
        node_projection_input = node_dim
        if fusion_mode == "projected_gated":
            blocks = self._input_block_ranges()
            if len(blocks) < 2:
                raise ValueError("projected_gated fusion requires at least two non-empty modalities.")
            for name, (start, stop) in blocks.items():
                self.modality_projectors[name] = nn.Sequential(
                    nn.Linear(stop - start, modality_projection_dim),
                    nn.LayerNorm(modality_projection_dim),
                    nn.ReLU(),
                )
            self.modality_gate_logits = nn.Parameter(torch.zeros(len(blocks)))
            node_projection_input = modality_projection_dim * len(blocks)

        self.node_proj = nn.Linear(node_projection_input, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        if node_transformation == "mlp":
            nn_module: nn.Module = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, latent_dim),
            )
        else:
            nn_module = nn.Linear(hidden_dim, latent_dim)

        self.encoder_conv = GINEConv(nn_module, edge_dim=hidden_dim, aggr=gine_aggregation)

        self.structure_decoder = None
        if structure_decoder == "mlp":
            self.structure_decoder = nn.Sequential(
                nn.Linear(latent_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        self.node_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, recon_dim),
        )
        self.edge_decoder = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_dim),
        )

    def node_reconstruction_target(self, x_norm: torch.Tensor) -> torch.Tensor:
        """Select the decoder target columns from BatchNorm-scaled node features."""
        index = self.node_recon_index
        if index.device != x_norm.device:
            index = index.to(device=x_norm.device)
            self.node_recon_index = index
        return x_norm.index_select(1, index)

    def _input_block_ranges(self) -> dict[str, tuple[int, int]]:
        ranges: dict[str, tuple[int, int]] = {}
        cursor = 0
        if self.tfidf_dim:
            ranges["tfidf"] = (cursor, cursor + self.tfidf_dim)
            cursor += self.tfidf_dim
        if self.sbert_dim:
            ranges["sbert"] = (cursor, cursor + self.sbert_dim)
            cursor += self.sbert_dim
        if self.extra_dim:
            ranges["extras"] = (cursor, cursor + self.extra_dim)
        return ranges

    def project_node_inputs(self, x_norm: torch.Tensor) -> torch.Tensor:
        """Legacy concat or gated, equally sized modality projections."""
        if self.fusion_mode == "concat":
            return x_norm
        assert self.modality_gate_logits is not None
        gates = torch.softmax(self.modality_gate_logits, dim=0)
        projected = []
        for gate, (name, (start, stop)) in zip(
            gates, self._input_block_ranges().items(), strict=True
        ):
            projected.append(gate * self.modality_projectors[name](x_norm[:, start:stop]))
        return torch.cat(projected, dim=1)

    def node_block_errors(
        self, x_rec: torch.Tensor, target: torch.Tensor
    ) -> dict[str, torch.Tensor]:
        """Per-node MSE for each reconstructed modality block."""
        squared = F.mse_loss(x_rec, target, reduction="none")
        original = self.node_recon_index.to(squared.device)
        result: dict[str, torch.Tensor] = {}
        for name, (start, stop) in self._input_block_ranges().items():
            positions = torch.nonzero(
                (original >= start) & (original < stop), as_tuple=False
            ).reshape(-1)
            if positions.numel():
                result[name] = squared.index_select(1, positions).mean(dim=1)
        return result

    def node_reconstruction_loss(
        self, x_rec: torch.Tensor, target: torch.Tensor
    ) -> torch.Tensor:
        if self.node_loss_mode == "global":
            return F.mse_loss(x_rec, target)
        blocks = self.node_block_errors(x_rec, target)
        weighted = [
            float(self.node_block_weights[name]) * values.mean()
            for name, values in blocks.items()
            if float(self.node_block_weights[name]) > 0
        ]
        weight_sum = sum(
            float(self.node_block_weights[name])
            for name in blocks
            if float(self.node_block_weights[name]) > 0
        )
        if not weighted or weight_sum <= 0:
            raise ValueError("At least one reconstructed node block must have positive weight.")
        return torch.stack(weighted).sum() / weight_sum

    def node_anomaly_error(self, x_rec: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Per-node counterpart of the configured training objective."""
        if self.node_loss_mode == "global":
            return F.mse_loss(x_rec, target, reduction="none").mean(dim=1)
        blocks = self.node_block_errors(x_rec, target)
        weighted = [
            float(self.node_block_weights[name]) * values
            for name, values in blocks.items()
            if float(self.node_block_weights[name]) > 0
        ]
        weight_sum = sum(
            float(self.node_block_weights[name])
            for name in blocks
            if float(self.node_block_weights[name]) > 0
        )
        return torch.stack(weighted).sum(dim=0) / weight_sum

    def standardize_inputs(
        self,
        x: torch.Tensor,
        edge_attr: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Apply node BN; pass edge features through unchanged (pre-normalised)."""
        x_norm = self.raw_node_norm(x)
        if edge_attr is not None and edge_attr.numel() > 0 and edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(1)
        return x_norm, edge_attr

    def encode(
        self,
        x_norm: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr_norm: torch.Tensor | None,
    ) -> torch.Tensor:
        x_h = self.node_proj(self.project_node_inputs(x_norm))
        edge_h = self.edge_proj(edge_attr_norm) if edge_attr_norm is not None else None
        return self.encoder_conv(x_h, edge_index, edge_h)

    def decode_structure(self, z: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """Directed (MLP) or symmetric (inner-product) logits for edge pairs."""
        src, dst = edge_index
        if self.structure_decoder is None:
            return (z[src] * z[dst]).sum(dim=1)
        return self.structure_decoder(torch.cat([z[src], z[dst]], dim=-1)).squeeze(-1)

    def decode_node_features(self, z: torch.Tensor) -> torch.Tensor:
        return self.node_decoder(z)

    def decode_edge_attributes(self, z: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        src, dst = edge_index
        return self.edge_decoder(torch.cat([z[src], z[dst]], dim=-1))

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
        """Return ``(z, x_norm, edge_attr_norm)`` for use in loss computation."""
        x_norm, edge_attr_norm = self.standardize_inputs(x, edge_attr)
        z = self.encode(x_norm, edge_index, edge_attr_norm)
        return z, x_norm, edge_attr_norm


def per_graph_negative_sampling(edge_index: torch.Tensor, ptr: torch.Tensor) -> torch.Tensor:
    """Sample one non-edge per observed edge, restricted to that graph's nodes.

    Mini-batch ``negative_sampling(..., num_nodes=batch.num_nodes)`` treats the
    disjoint union as one graph and yields trivial cross-graph negatives.
    """
    if edge_index.size(1) == 0 or ptr.numel() < 2:
        return edge_index.new_zeros((2, 0))
    src, dst = edge_index[0], edge_index[1]
    chunks: list[torch.Tensor] = []
    n_graphs = int(ptr.numel() - 1)
    for graph in range(n_graphs):
        lo = int(ptr[graph].item())
        hi = int(ptr[graph + 1].item())
        n_nodes = hi - lo
        mask = (src >= lo) & (src < hi)
        pos = edge_index[:, mask]
        if pos.size(1) == 0 or n_nodes <= 0:
            continue
        local = pos - lo
        neg_local = negative_sampling(
            local,
            num_nodes=n_nodes,
            num_neg_samples=pos.size(1),
        )
        if neg_local.numel() == 0:
            continue
        chunks.append(neg_local + lo)
    if not chunks:
        return edge_index.new_zeros((2, 0))
    return torch.cat(chunks, dim=1)


def complete_directed_index(num_nodes: int, device: torch.device) -> torch.Tensor:
    """All directed pairs including self-loops (collapsed graphs may loop)."""
    src = torch.arange(num_nodes, device=device).repeat_interleave(num_nodes)
    dst = torch.arange(num_nodes, device=device).repeat(num_nodes)
    return torch.stack([src, dst], dim=0)


def graph_ptr(batch, num_graphs: int, device: torch.device) -> torch.Tensor:
    """Node-offset pointer tensor, reconstructed from ``batch.batch`` if needed."""
    ptr = getattr(batch, "ptr", None)
    if ptr is not None:
        return ptr
    counts = torch.bincount(batch.batch, minlength=num_graphs)
    out = torch.zeros(num_graphs + 1, dtype=torch.long, device=device)
    out[1:] = torch.cumsum(counts, dim=0)
    return out


def _structure_terms(
    model: AttributeAwareGAE,
    z: torch.Tensor,
    pos_index: torch.Tensor,
    neg_index: torch.Tensor | None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (mean pos BCE, mean neg BCE) as logits; zeros when a set is empty."""
    device = z.device
    pos_loss = torch.tensor(0.0, device=device)
    neg_loss = torch.tensor(0.0, device=device)
    if pos_index.size(1) > 0:
        pos_logits = model.decode_structure(z, pos_index)
        pos_loss = F.binary_cross_entropy_with_logits(
            pos_logits, torch.ones_like(pos_logits)
        )
    if neg_index is not None and neg_index.size(1) > 0:
        neg_logits = model.decode_structure(z, neg_index)
        neg_loss = F.binary_cross_entropy_with_logits(
            neg_logits, torch.zeros_like(neg_logits)
        )
    return pos_loss, neg_loss


def _structure_error_vector(
    model: AttributeAwareGAE,
    z: torch.Tensor,
    edge_index: torch.Tensor,
    ptr: torch.Tensor,
    *,
    include_non_edges: bool,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    """Per-graph structure error: pos BCE (+ in-graph non-edge BCE when requested)."""
    device = z.device
    num_graphs = int(ptr.numel() - 1)
    scores = torch.zeros(num_graphs, device=device)
    src = edge_index[0]
    for graph in range(num_graphs):
        lo = int(ptr[graph].item())
        hi = int(ptr[graph + 1].item())
        n_nodes = hi - lo
        mask = (src >= lo) & (src < hi) if edge_index.size(1) else None
        pos = edge_index[:, mask] if mask is not None else edge_index.new_zeros((2, 0))
        neg = None
        if include_non_edges and n_nodes > 0:
            neg = _in_graph_non_edges(pos, lo, n_nodes, device, generator)
        pos_loss, neg_loss = _structure_terms(model, z, pos, neg)
        scores[graph] = pos_loss + (neg_loss if include_non_edges else torch.tensor(0.0, device=device))
    return scores


def _in_graph_non_edges(
    pos: torch.Tensor,
    lo: int,
    n_nodes: int,
    device: torch.device,
    generator: torch.Generator | None,
) -> torch.Tensor:
    """Non-edges inside one graph: full digraph when small, else sampled."""
    del generator  # sampling uses PyG's RNG; seed the process for campaigns
    if n_nodes <= 0:
        return pos.new_zeros((2, 0))
    if n_nodes <= FULL_STRUCTURE_MAX_NODES:
        complete = complete_directed_index(n_nodes, device)
        adj = torch.zeros((n_nodes, n_nodes), dtype=torch.bool, device=device)
        if pos.size(1):
            adj[pos[0] - lo, pos[1] - lo] = True
        keep = ~adj[complete[0], complete[1]]
        return complete[:, keep] + lo
    local = pos - lo if pos.size(1) else pos.new_zeros((2, 0))
    n_pos = max(int(pos.size(1)), 1)
    neg_local = negative_sampling(
        local,
        num_nodes=n_nodes,
        num_neg_samples=n_pos,
        force_undirected=False,
    )
    if neg_local.numel() == 0:
        return pos.new_zeros((2, 0))
    return neg_local + lo


def train_epoch(
    model: AttributeAwareGAE,
    loader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    *,
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
) -> tuple[float, float, float, float]:
    """Run one full training epoch with per-graph structure negatives."""
    model.train()
    total_loss = total_str = total_node = total_edge = 0.0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
        num_graphs = batch.num_graphs if hasattr(batch, "num_graphs") else 1
        ptr = graph_ptr(batch, num_graphs, device)

        loss_str = torch.tensor(0.0, device=device)
        if batch.edge_index.size(1) > 0:
            pos_logits = model.decode_structure(z, batch.edge_index)
            neg_edge = per_graph_negative_sampling(batch.edge_index, ptr)
            pos_loss = F.binary_cross_entropy_with_logits(
                pos_logits, torch.ones_like(pos_logits)
            )
            if neg_edge.size(1) > 0:
                neg_logits = model.decode_structure(z, neg_edge)
                neg_loss = F.binary_cross_entropy_with_logits(
                    neg_logits, torch.zeros_like(neg_logits)
                )
            else:
                neg_loss = torch.tensor(0.0, device=device)
            loss_str = pos_loss + neg_loss

        x_rec = model.decode_node_features(z)
        loss_node = model.node_reconstruction_loss(
            x_rec, model.node_reconstruction_target(x_norm)
        )

        loss_edge = torch.tensor(0.0, device=device)
        if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
            edge_rec = model.decode_edge_attributes(z, batch.edge_index)
            loss_edge = F.mse_loss(edge_rec, edge_attr_norm)

        loss = alpha * loss_str + beta * loss_node + gamma * loss_edge
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n = batch.num_graphs
        total_loss += loss.item() * n
        total_str += loss_str.item() * n
        total_node += loss_node.item() * n
        total_edge += loss_edge.item() * n

    ng = len(loader.dataset)
    if ng == 0:
        return 0.0, 0.0, 0.0, 0.0
    return total_loss / ng, total_str / ng, total_node / ng, total_edge / ng


@torch.no_grad()
def compute_anomaly_scores(
    model: AttributeAwareGAE,
    loader,
    device: torch.device,
    *,
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
    return_components: bool = False,
    include_structure_non_edges: bool = True,
) -> tuple:
    """Compute per-graph anomaly scores (weighted reconstruction error).

    Structure error matches training: BCE on observed edges plus in-graph
    non-edges (full directed adjacency on small collapsed graphs). Combined
    scores stay ``α·str + β·node + γ·edge``.
    """
    model.eval()
    all_scores, all_labels = [], []
    all_structure, all_node, all_edge = [], [], []
    all_tfidf, all_sbert, all_extras = [], [], []
    graph_ids: list[str] = []

    for batch in loader:
        batch = batch.to(device)
        z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
        num_graphs = batch.num_graphs if hasattr(batch, "num_graphs") else 1
        ptr = graph_ptr(batch, num_graphs, device)

        g_str = _structure_error_vector(
            model,
            z,
            batch.edge_index,
            ptr,
            include_non_edges=include_structure_non_edges,
        )

        x_rec = model.decode_node_features(z)
        target = model.node_reconstruction_target(x_norm)
        node_errors = model.node_anomaly_error(x_rec, target)
        g_node = scatter(
            node_errors, batch.batch, dim=0, reduce="mean", dim_size=num_graphs
        )

        g_edge = torch.zeros(num_graphs, device=device)
        if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
            ea_errors = F.mse_loss(
                model.decode_edge_attributes(z, batch.edge_index),
                edge_attr_norm,
                reduction="none",
            ).mean(dim=1)
            edge_batch = batch.batch[batch.edge_index[0]]
            g_edge = scatter(
                ea_errors, edge_batch, dim=0, reduce="mean", dim_size=num_graphs
            )

        g_str = torch.nan_to_num(g_str, 0.0)
        g_edge = torch.nan_to_num(g_edge, 0.0)

        total = alpha * g_str + beta * g_node + gamma * g_edge
        all_scores.append(total.cpu())
        all_labels.append(batch.y.cpu())
        if return_components:
            all_structure.append(g_str.cpu())
            all_node.append(g_node.cpu())
            all_edge.append(g_edge.cpu())
            block_errors = model.node_block_errors(x_rec, target)
            for name, destination in (
                ("tfidf", all_tfidf),
                ("sbert", all_sbert),
                ("extras", all_extras),
            ):
                values = block_errors.get(name)
                if values is None:
                    destination.append(torch.zeros(num_graphs))
                else:
                    destination.append(
                        scatter(
                            values, batch.batch, dim=0, reduce="mean", dim_size=num_graphs
                        ).cpu()
                    )
            graph_ids.extend(_batch_graph_ids(batch, num_graphs))

    scores = torch.cat(all_scores).numpy()
    labels = torch.cat(all_labels).numpy()
    if not return_components:
        return scores, labels
    zeros = scores * 0.0
    components = {
        "structure": torch.cat(all_structure).numpy() if all_structure else zeros,
        "node": torch.cat(all_node).numpy() if all_node else zeros,
        "edge": torch.cat(all_edge).numpy() if all_edge else zeros,
        "tfidf": torch.cat(all_tfidf).numpy() if all_tfidf else zeros,
        "sbert": torch.cat(all_sbert).numpy() if all_sbert else zeros,
        "extras": torch.cat(all_extras).numpy() if all_extras else zeros,
    }
    return scores, labels, components, graph_ids


def _batch_graph_ids(batch, num_graphs: int) -> list[str]:
    """Return per-graph identifiers from a PyG batch when available."""
    for attribute in ("block_id", "window_id", "sequence_id", "graph_id"):
        if hasattr(batch, attribute):
            raw = getattr(batch, attribute)
            if raw is None:
                continue
            if hasattr(raw, "tolist"):
                values = raw.tolist()
            elif isinstance(raw, (list, tuple)):
                values = list(raw)
            else:
                values = [raw]
            if len(values) == num_graphs:
                return [str(value) for value in values]
    return [str(index) for index in range(num_graphs)]


# ---------------------------------------------------------------------------
# Campaign helpers inlined from run_ablation.py, dataset.py, ablation.py,
# classical_baseline.py, and utils.py. No src.modules imports.
# ---------------------------------------------------------------------------

# Inlined helpers from run_ablation.py, src/modules/ablation.py,
# src/modules/dataset.py, src/modules/classical_baseline.py, src/modules/utils.py.
# This fragment is concatenated into Ablation_Full_Monty.ipynb and then deleted.


import copy
import gzip
import json
import random
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:  # Colab runtimes sometimes omit PyYAML
    yaml = None


BGL_SPLIT_STRIDE = 1 << 40
NODE_EXTRA_DIM = 9
SBERT_EMBED_DIM = 384

TRAINING = {
    "train_mode": "clean",
    "test_run": False,
    "test_samples": 5000,
    "hidden_dim": 128,
    "latent_dim": 64,
    "batch_size": 256,
    "epochs": 25,
    "learning_rate": 0.001,
    "alpha": 1.0,
    "beta": 1.0,
    "gamma": 1.0,
    "pre_normalize_edges": True,
    "minimum_edge_std": 0.1,
}

GAE_ARCH = {
    "gine_aggregation": "sum",
    "node_transformation": "mlp",
    "structure_decoder": "mlp",
    "node_reconstruct": "all",
    "fusion_mode": "concat",
    "modality_projection_dim": 64,
    "node_loss_mode": "global",
    "node_block_weights": {"tfidf": 1.0, "sbert": 1.0, "extras": 1.0},
}

EXPECTED_ARMS = (
    "baseline_full",
    "gine_mean_agg",
    "gine_max_agg",
    "linear_node_transform",
    "latent_32",
    "hidden_256",
    "learning_rate_001",
    "gamma_05",
    "inner_product_structure",
    "alpha_0",
    "beta_0",
    "gamma_0",
    "lexical_node_recon",
)



def resolved_arm(arm: str) -> tuple[dict[str, Any], dict[str, Any]]:
    if arm not in ARM_OVERRIDES:
        raise KeyError(arm)
    training = copy.deepcopy(TRAINING)
    arch = copy.deepcopy(GAE_ARCH)
    for key, value in ARM_OVERRIDES[arm].items():
        if key in training:
            training[key] = value
        elif key in arch:
            arch[key] = value
        else:
            raise KeyError(f"Unknown Family B override {key!r} on arm {arm!r}")
    if SMOKE:
        training["test_run"] = True
        training["epochs"] = 1
    return training, arch


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _safe_name(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "-", value.strip())
    if not cleaned:
        raise ValueError("Run names must contain at least one letter or number.")
    return cleaned.strip(".-")


def _rounded(value: float) -> float:
    return round(float(value), 6)


def graph_split_dir(path: str | Path) -> Path:
    path = Path(path)
    name = path.name
    if name.endswith(".gz"):
        name = name[:-3]
    stem = Path(name).stem
    return path.parent / f"{stem}_splits"


def gunzip_file(source: str | Path, destination: str | Path | None = None) -> Path:
    source = Path(source)
    if destination is None:
        if source.suffix != ".gz":
            return source
        destination = source.with_suffix("")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(source, "rb") as incoming, destination.open("wb") as outgoing:
        shutil.copyfileobj(incoming, outgoing, length=16 * 1024 * 1024)
    return destination


def load_bundle_meta(graph_path: str | Path) -> dict[str, Any] | None:
    graph_path = Path(graph_path)
    sidecar = graph_path.with_name("dataset_meta.json")
    if not sidecar.exists() and graph_path.suffix == ".gz":
        sidecar = graph_path.with_suffix("").with_name("dataset_meta.json")
    if sidecar.exists():
        return json.loads(sidecar.read_text())
    parent_meta = graph_path.parent / "dataset_meta.json"
    if parent_meta.exists():
        return json.loads(parent_meta.read_text())
    try:
        bundle = torch.load(graph_path, map_location="cpu", weights_only=False)
    except Exception:
        return None
    if not isinstance(bundle, dict):
        return None
    meta = bundle.get("dataset_meta")
    if isinstance(meta, dict):
        return meta
    extracted = {
        key: bundle[key]
        for key in (
            "node_dim",
            "edge_dim",
            "embed_dim",
            "feature_contract",
            "llm_enrichment_enabled",
            "enrichment_model_size",
            "use_edge_features",
            "embedding_flags",
            "graph_identity",
            "split_lock_id",
        )
        if key in bundle
    }
    return extracted or None


def load_graph_splits(path: str | Path) -> tuple[list, list, list, dict[str, Any]]:
    path = Path(path)
    split_dir = graph_split_dir(path)
    train_path = split_dir / "train.pt"
    if train_path.exists() and (split_dir / "val.pt").exists() and (split_dir / "test.pt").exists():
        train_graphs = torch.load(train_path, weights_only=False, map_location="cpu")
        val_graphs = torch.load(split_dir / "val.pt", weights_only=False, map_location="cpu")
        test_graphs = torch.load(split_dir / "test.pt", weights_only=False, map_location="cpu")
        meta: dict[str, Any] = {}
        meta_path = split_dir / "meta.json"
        if meta_path.exists():
            meta = json.loads(meta_path.read_text())
        sidecar = path.with_name("dataset_meta.json")
        if sidecar.exists():
            meta.setdefault("dataset_meta", json.loads(sidecar.read_text()))
        return list(train_graphs), list(val_graphs), list(test_graphs), meta

    bundle = torch.load(path, weights_only=False, map_location="cpu")
    all_data = bundle["data_list"]
    train_graphs = [all_data[int(i)] for i in bundle["idx_train"]]
    val_graphs = [all_data[int(i)] for i in bundle["idx_val"]]
    test_graphs = [all_data[int(i)] for i in bundle["idx_test"]]
    meta = {
        "node_dim": bundle.get("node_dim"),
        "edge_dim": bundle.get("edge_dim"),
        "embed_dim": bundle.get("embed_dim"),
        "dataset_meta": bundle.get("dataset_meta") or {},
    }
    return train_graphs, val_graphs, test_graphs, meta


def sequence_ids_from_graphs(data_list: list) -> list[str]:
    ids: list[str] = []
    for index, graph in enumerate(data_list):
        value = None
        for attribute in ("block_id", "window_id", "sequence_id", "graph_id"):
            if hasattr(graph, attribute):
                raw = getattr(graph, attribute)
                value = raw.item() if hasattr(raw, "item") else raw
                break
        ids.append(str(index if value is None else value))
    return ids


def embedding_block_dims(
    embed_dim: int,
    *,
    tfidf_enabled: bool,
    sbert_enabled: bool,
    sbert_width: int = SBERT_EMBED_DIM,
) -> tuple[int, int]:
    if tfidf_enabled and sbert_enabled:
        if embed_dim <= sbert_width:
            raise ValueError(
                f"Hybrid embed_dim={embed_dim} is too small for MiniLM width {sbert_width}."
            )
        return int(embed_dim - sbert_width), int(sbert_width)
    if sbert_enabled:
        return 0, int(embed_dim)
    return int(embed_dim), 0


def sbert_dim_from_meta(meta: Mapping[str, Any] | None, node_dim: int | None = None) -> int:
    if not meta:
        return 0
    if meta.get("sbert_dim") is not None:
        return int(meta["sbert_dim"])
    flags = meta.get("embedding_flags") or {}
    identity = meta.get("graph_identity") or {}
    sbert_enabled = bool(flags.get("sbert_enabled", identity.get("sbert_enabled", False)))
    tfidf_enabled = bool(flags.get("tfidf_enabled", identity.get("tfidf_enabled", True)))
    embed_dim = int(meta.get("embed_dim") or 0)
    if embed_dim <= 0:
        width = int(node_dim or meta.get("node_dim") or 0)
        embed_dim = max(width - NODE_EXTRA_DIM, 0)
    if embed_dim <= 0 or not sbert_enabled:
        return 0
    _, sbert_dim = embedding_block_dims(
        embed_dim, tfidf_enabled=tfidf_enabled, sbert_enabled=True
    )
    return sbert_dim


def flatten_split_meta(
    split_meta: Mapping[str, Any] | None,
    bundle_meta: Mapping[str, Any] | None,
) -> dict[str, Any]:
    merged: dict[str, Any] = {}
    if isinstance(split_meta, Mapping):
        nested = split_meta.get("dataset_meta")
        if isinstance(nested, Mapping):
            merged.update(dict(nested))
        for key, value in split_meta.items():
            if key != "dataset_meta" and value is not None:
                merged.setdefault(key, value)
    if isinstance(bundle_meta, Mapping):
        for key, value in bundle_meta.items():
            merged.setdefault(key, value)
    return merged


def window_feature_matrices(
    train_graphs: list[Any],
    *splits: list[Any],
    oov_cluster_id: int = -1,
) -> tuple[np.ndarray, ...]:
    vocabulary = sorted(
        {
            int(cluster_id)
            for graph in train_graphs
            for cluster_id in _cluster_ids(graph)
            if int(cluster_id) >= 0
        }
    )
    if not vocabulary:
        raise ValueError("Isolation Forest baseline found no known training templates.")
    index = {cluster_id: position for position, cluster_id in enumerate(vocabulary)}

    def transform(graphs: list[Any]) -> np.ndarray:
        matrix = np.zeros((len(graphs), len(vocabulary) + 2), dtype=np.float32)
        for row, graph in enumerate(graphs):
            cluster_ids = _cluster_ids(graph)
            n_events = len(cluster_ids)
            if not n_events:
                continue
            for cluster_id in cluster_ids:
                if cluster_id in index:
                    matrix[row, index[cluster_id]] += 1.0
            matrix[row, -2] = sum(cluster_id == oov_cluster_id for cluster_id in cluster_ids) / n_events
            matrix[row, -1] = float(np.log1p(n_events))
        return matrix

    return tuple(transform(graphs) for graphs in (train_graphs, *splits))


def _cluster_ids(graph: Any) -> list[int]:
    value = getattr(graph, "event_cluster_ids", None)
    if value is None:
        raise ValueError(
            "Graph bundle lacks event_cluster_ids required by Isolation Forest. "
            "Rebuild the BGL campaign graphs with the current pipeline."
        )
    if hasattr(value, "detach"):
        value = value.detach().cpu().tolist()
    elif hasattr(value, "tolist"):
        value = value.tolist()
    return [int(cluster_id) for cluster_id in value]


def _copy_graph_splits(source_graph: Path, dest_graph: Path) -> None:
    source_dir = source_graph.parent / "graph_dataset_splits"
    if not source_dir.exists():
        source_dir = graph_split_dir(source_graph)
    if not source_dir.exists():
        return
    destination = graph_split_dir(dest_graph)
    if destination.resolve() == source_dir.resolve():
        return
    shutil.copytree(source_dir, destination, dirs_exist_ok=True)


def _normalise_edge_attributes(
    train_graphs: list,
    val_graphs: list,
    test_graphs: list,
    *,
    enabled: bool,
    minimum_std: float,
) -> tuple[list[float] | None, list[float] | None]:
    if not enabled:
        return None, None
    training_edges = [
        graph.edge_attr.float()
        for graph in train_graphs
        if getattr(graph, "edge_attr", None) is not None and graph.edge_attr.numel() > 0
    ]
    if not training_edges:
        return None, None
    stacked = torch.cat(training_edges, dim=0)
    mean = stacked.mean(dim=0)
    std = stacked.std(dim=0).clamp_min(minimum_std)
    for graph in [*train_graphs, *val_graphs, *test_graphs]:
        edge_attr = getattr(graph, "edge_attr", None)
        if edge_attr is not None and edge_attr.numel() > 0:
            graph.edge_attr = (edge_attr.float() - mean) / std
    return mean.tolist(), std.tolist()


def _loss_history(history: list[dict[str, float | int]]) -> dict[str, list[float]]:
    return {
        "total": [float(entry["train_total_loss"]) for entry in history],
        "structure": [float(entry["train_structure_loss"]) for entry in history],
        "node": [float(entry["train_node_loss"]) for entry in history],
        "edge": [float(entry["train_edge_loss"]) for entry in history],
    }


def _threshold_and_metrics(
    labels: np.ndarray, scores: np.ndarray, *, threshold: float | None = None
) -> tuple[float, dict[str, float]]:
    from sklearn.metrics import precision_recall_curve

    if threshold is None:
        precision, recall, thresholds = precision_recall_curve(labels, scores)
        if len(thresholds) == 0:
            threshold = float(np.median(scores))
        else:
            f1_values = (2 * precision[:-1] * recall[:-1]) / (
                precision[:-1] + recall[:-1] + 1e-12
            )
            threshold = float(thresholds[int(np.nanargmax(f1_values))])
    return threshold, _score_metrics(labels, scores, threshold)


def _score_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, float]:
    from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

    predictions = (scores > threshold).astype(int)
    has_both_classes = len(np.unique(labels)) == 2
    return {
        "f1": _rounded(f1_score(labels, predictions, zero_division=0)),
        "pr_auc": _rounded(average_precision_score(labels, scores)) if has_both_classes else 0.0,
        "roc_auc": _rounded(roc_auc_score(labels, scores)) if has_both_classes else 0.0,
    }


def _node_recon_index_for_bundle(
    *,
    node_dim: int,
    node_reconstruct: str,
    split_meta: dict[str, Any],
    bundle_meta: dict[str, Any] | None,
) -> Any:
    if node_reconstruct != "without_sbert":
        return torch.arange(node_dim, dtype=torch.long)
    meta = flatten_split_meta(split_meta, bundle_meta)
    sbert_dim = sbert_dim_from_meta(meta, node_dim=node_dim)
    return node_recon_index_tensor(
        node_dim,
        sbert_dim=sbert_dim,
        extra_dim=int(meta.get("node_extra_dim", 9)),
    )


def _dump_config(path: Path, config: Mapping[str, Any]) -> None:
    serialisable = {key: value for key, value in config.items() if key != "__pipeline__"}
    if yaml is not None:
        path.write_text(yaml.safe_dump(serialisable, sort_keys=False))
    else:
        path.write_text(json.dumps(serialisable, indent=2, default=str))


def _component_aucs(labels: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    from sklearn.metrics import average_precision_score, roc_auc_score

    has_both = len(np.unique(labels)) == 2
    return {
        "pr_auc": float(average_precision_score(labels, scores)) if has_both else 0.0,
        "roc_auc": float(roc_auc_score(labels, scores)) if has_both else 0.0,
    }


def component_metrics(
    labels: np.ndarray,
    *,
    combined: np.ndarray,
    structure: np.ndarray,
    node: np.ndarray,
    edge: np.ndarray,
    predictions: np.ndarray,
    alpha: float,
    beta: float,
    gamma: float,
    node_blocks: Mapping[str, np.ndarray] | None = None,
) -> dict[str, Any]:
    metrics: dict[str, Any] = {
        "combined": _component_aucs(labels, combined),
        "structure": _component_aucs(labels, structure),
        "node": _component_aucs(labels, node),
        "edge": _component_aucs(labels, edge),
    }
    for name, values in (node_blocks or {}).items():
        array = np.asarray(values)
        metrics[name] = {
            **_component_aucs(labels, array),
            "mean_normal": float(array[labels == 0].mean()) if (labels == 0).any() else 0.0,
            "mean_anomaly": float(array[labels == 1].mean()) if (labels == 1).any() else 0.0,
        }
    weighted = np.column_stack([alpha * structure, beta * node, gamma * edge])
    totals = weighted.sum(axis=1, keepdims=True)
    shares = np.divide(
        weighted,
        np.maximum(totals, 1e-12),
        out=np.zeros_like(weighted, dtype=float),
        where=totals > 0,
    )
    true_positives = (labels == 1) & (predictions == 1)
    names = ("structure", "node", "edge")
    if true_positives.any():
        mean_share = shares[true_positives].mean(axis=0)
        dominant = np.argmax(shares[true_positives], axis=1)
        metrics["true_positive_share"] = {
            name: float(mean_share[index]) for index, name in enumerate(names)
        }
        metrics["true_positive_dominance_count"] = {
            name: int((dominant == index).sum()) for index, name in enumerate(names)
        }
        metrics["n_true_positives"] = int(true_positives.sum())
    else:
        metrics["true_positive_share"] = {"structure": 0.0, "node": 0.0, "edge": 0.0}
        metrics["true_positive_dominance_count"] = {"structure": 0, "node": 0, "edge": 0}
        metrics["n_true_positives"] = 0
    return metrics


def write_eval_pack(
    output_dir: str | Path,
    *,
    metrics: Mapping[str, Any],
    history_epochs: list[dict[str, Any]],
    labels: np.ndarray,
    scores: np.ndarray,
    structure: np.ndarray,
    node: np.ndarray,
    edge: np.ndarray,
    threshold: float,
    alpha: float,
    beta: float,
    gamma: float,
    node_blocks: Mapping[str, np.ndarray] | None = None,
    graph_ids: list[str] | None = None,
    config: Mapping[str, Any] | None = None,
    campaign_meta: Mapping[str, Any] | None = None,
) -> dict[str, Path]:
    output_dir = Path(output_dir)
    figures = output_dir / "figures"
    scores_dir = output_dir / "scores"
    figures.mkdir(parents=True, exist_ok=True)
    scores_dir.mkdir(parents=True, exist_ok=True)

    predictions = (np.asarray(scores) > threshold).astype(int)
    labels = np.asarray(labels)
    structure = np.asarray(structure)
    node = np.asarray(node)
    edge = np.asarray(edge)
    components = component_metrics(
        labels,
        combined=np.asarray(scores),
        structure=structure,
        node=node,
        edge=edge,
        predictions=predictions,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        node_blocks=node_blocks,
    )
    component_path = output_dir / "component_metrics.json"
    component_path.write_text(json.dumps(components, indent=2))

    history_path = output_dir / "history.json"
    history_path.write_text(json.dumps({"epochs": history_epochs}, indent=2))

    config_path = output_dir / "config.yaml"
    if config is not None:
        _dump_config(config_path, config)

    score_columns: dict[str, Any] = {
        "graph_id": graph_ids if graph_ids is not None else list(range(len(labels))),
        "label": labels.astype(int),
        "prediction": predictions,
        "score": np.asarray(scores),
        "structure": structure,
        "node": node,
        "edge": edge,
        "weighted_structure": alpha * structure,
        "weighted_node": beta * node,
        "weighted_edge": gamma * edge,
    }
    for name, values in (node_blocks or {}).items():
        score_columns[name] = np.asarray(values)
    score_frame = pd.DataFrame(score_columns)
    scores_csv = scores_dir / "test_component_scores.csv"
    score_frame.to_csv(scores_csv, index=False)

    manifest_path = output_dir / "manifest.json"
    if campaign_meta is not None:
        manifest_path.write_text(json.dumps(dict(campaign_meta), indent=2, default=str))
    else:
        manifest_path.write_text("{}")

    written: dict[str, Path] = {
        "component_metrics": component_path,
        "history": history_path,
        "config": config_path,
        "scores_csv": scores_csv,
        "manifest": manifest_path,
        "metrics": output_dir / "metrics.json",
    }
    (output_dir / "metrics.json").write_text(json.dumps(dict(metrics), indent=2, default=str))
    written.update(
        _write_eval_figures(
            figures,
            history_epochs=history_epochs,
            labels=labels,
            scores=np.asarray(scores),
            predictions=predictions,
            threshold=threshold,
            structure=structure,
            node=node,
            edge=edge,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
        )
    )
    return written


def _write_eval_figures(
    figures: Path,
    *,
    history_epochs: list[dict[str, Any]],
    labels: np.ndarray,
    scores: np.ndarray,
    predictions: np.ndarray,
    threshold: float,
    structure: np.ndarray,
    node: np.ndarray,
    edge: np.ndarray,
    alpha: float,
    beta: float,
    gamma: float,
) -> dict[str, Path]:
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from sklearn.metrics import (
            ConfusionMatrixDisplay,
            average_precision_score,
            confusion_matrix,
            precision_recall_curve,
            roc_auc_score,
            roc_curve,
        )
    except ImportError:
        return {}

    written: dict[str, Path] = {}
    if history_epochs:
        epochs = [int(entry["epoch"]) for entry in history_epochs]
        figure, axes = plt.subplots(1, 3, figsize=(18, 4.5))
        axes[0].plot(epochs, [entry["train_total_loss"] for entry in history_epochs], color="black", linewidth=2)
        axes[0].set(title="Total training loss", xlabel="Epoch", ylabel="Loss")
        for key, color, label in (
            ("train_structure_loss", "#4C72B0", "structure"),
            ("train_node_loss", "#55A868", "node"),
            ("train_edge_loss", "#C44E52", "edge"),
        ):
            axes[1].plot(epochs, [entry[key] for entry in history_epochs], linewidth=2, color=color, label=label)
        axes[1].set(title="Reconstruction components", xlabel="Epoch", ylabel="Loss")
        axes[1].legend()
        if "val_f1" in history_epochs[0]:
            axes[2].plot(epochs, [entry["val_f1"] for entry in history_epochs], linewidth=2, color="#8172B2", label="Val F1")
            axes[2].plot(epochs, [entry["val_pr_auc"] for entry in history_epochs], linewidth=1.5, color="#DD8452", label="Val PR-AUC")
            axes[2].plot(epochs, [entry["val_roc_auc"] for entry in history_epochs], linewidth=1.5, color="#937860", label="Val ROC-AUC")
            axes[2].set(title="Validation metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1.05))
            axes[2].legend()
        for axis in axes:
            axis.grid(alpha=0.25)
        figure.tight_layout()
        path = figures / "training_history.png"
        figure.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        written["training_history"] = path
        learning_path = figures / "learning_curve.png"
        shutil.copy2(path, learning_path)
        written["learning_curve"] = learning_path

    figure, axes = plt.subplots(1, 1, figsize=(7, 4.5))
    for class_id, color, title in ((0, "#4C72B0", "Normal"), (1, "#C44E52", "Anomaly")):
        subset = scores[labels == class_id]
        if len(subset):
            axes.hist(subset, density=True, bins=40, alpha=0.45, color=color, label=f"{title} (n={len(subset)})")
    axes.axvline(
        threshold,
        color="black",
        linestyle="--",
        linewidth=2,
        label=f"threshold = {float(threshold):.4f}",
    )
    axes.set(
        title="Normal vs anomaly score distribution",
        xlabel="Anomaly score (weighted reconstruction error)",
        ylabel="Density",
    )
    axes.legend()
    axes.grid(alpha=0.25)
    figure.tight_layout()
    dist_path = figures / "test_score_distribution.png"
    figure.savefig(dist_path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    written["score_distribution"] = dist_path

    figure, axis = plt.subplots(figsize=(5, 4.5))
    ConfusionMatrixDisplay(
        confusion_matrix(labels, predictions, labels=[0, 1]),
        display_labels=["Normal", "Anomaly"],
    ).plot(ax=axis, colorbar=False, cmap="Blues")
    axis.set_title("Held-out confusion matrix")
    figure.tight_layout()
    cm_path = figures / "confusion_matrix.png"
    figure.savefig(cm_path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    written["confusion_matrix"] = cm_path

    if len(np.unique(labels)) == 2:
        figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        precision, recall, _ = precision_recall_curve(labels, scores)
        axes[0].plot(recall, precision, linewidth=2, label=f"PR-AUC = {average_precision_score(labels, scores):.4f}")
        axes[0].axhline(float(labels.mean()), color="gray", linestyle="--", label="positive-class rate")
        axes[0].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
        axes[0].legend()
        false_positive_rate, true_positive_rate, _ = roc_curve(labels, scores)
        axes[1].plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"ROC-AUC = {roc_auc_score(labels, scores):.4f}")
        axes[1].plot([0, 1], [0, 1], "--", color="gray", label="random")
        axes[1].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="False positive rate", ylabel="True positive rate", title="ROC curve")
        axes[1].legend()
        figure.tight_layout()
        pr_path = figures / "test_pr_roc.png"
        figure.savefig(pr_path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        written["pr_roc"] = pr_path

        pr_only, axis = plt.subplots(figsize=(6.5, 4.5))
        axis.plot(recall, precision, linewidth=2, label=f"PR-AUC = {average_precision_score(labels, scores):.4f}")
        axis.axhline(float(labels.mean()), color="gray", linestyle="--", label="positive-class rate")
        axis.set(xlim=(0, 1), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
        axis.legend()
        axis.grid(alpha=0.25)
        pr_only.tight_layout()
        pr_only_path = figures / "test_pr_curve.png"
        pr_only.savefig(pr_only_path, dpi=160, bbox_inches="tight")
        plt.close(pr_only)
        written["pr_curve"] = pr_only_path

    weighted = pd.DataFrame(
        {
            "Structure": alpha * structure,
            "Node": beta * node,
            "Edge": gamma * edge,
            "label": labels,
            "prediction": predictions,
        }
    )
    true_positives = weighted[(weighted.label == 1) & (weighted.prediction == 1)]
    if len(true_positives):
        contribution = true_positives[["Structure", "Node", "Edge"]].div(
            true_positives[["Structure", "Node", "Edge"]].sum(axis=1), axis=0
        ).fillna(0)
        figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        average_contribution = contribution.mean().mul(100)
        axes[0].pie(average_contribution, labels=list(average_contribution.index), autopct="%1.1f%%")
        axes[0].set_title(f"Average weighted contribution — true positives (n={len(true_positives)})")
        dominant = contribution.idxmax(axis=1).value_counts().reindex(["Structure", "Node", "Edge"], fill_value=0)
        axes[1].bar(dominant.index, dominant.values, color=["#4C72B0", "#55A868", "#C44E52"])
        axes[1].set(title="Dominant component per true positive", ylabel="Graphs")
        axes[1].grid(axis="y", alpha=0.25)
        figure.tight_layout()
        contrib_path = figures / "component_contribution.png"
        figure.savefig(contrib_path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        written["component_contribution"] = contrib_path
    return written


def resolve_campaign_baseline_name(
    records: list[Mapping[str, Any]] | pd.DataFrame | None = None,
    *,
    dataset: str,
    baseline_name: str | None = None,
) -> str:
    if baseline_name:
        return str(baseline_name)
    names: set[str] = set()
    if isinstance(records, pd.DataFrame):
        if "name" in records.columns:
            names = {str(item) for item in records["name"].tolist()}
    elif records:
        names = {str(item.get("name")) for item in records if item.get("name")}
    if "hybrid_llm" in names:
        return "hybrid_llm"
    if "baseline_full" in names:
        return "baseline_full"
    return "hybrid_llm" if str(dataset).lower() == "bgl" else "baseline_full"


def write_campaign_report(
    workspace: str | Path,
    *,
    dataset: str,
    campaign_id: str,
    baseline_name: str | None = None,
) -> Path:
    workspace = Path(workspace)
    output_root = workspace / "outputs" / dataset
    campaign_dir = output_root / "campaigns" / campaign_id
    campaign_dir.mkdir(parents=True, exist_ok=True)

    records: list[dict[str, Any]] = []
    prefix = f"{campaign_id}_"
    for metrics_path in sorted(output_root.glob("*/metrics.json")):
        run_dir = metrics_path.parent
        if run_dir.name in {"campaigns", "notebook_reports"} or run_dir.name.endswith("_ablation_matrix"):
            continue
        if not run_dir.name.startswith(prefix) and run_dir.name != campaign_id:
            manifest_path = run_dir / "manifest.json"
            if not manifest_path.exists():
                continue
            manifest = json.loads(manifest_path.read_text())
            if manifest.get("campaign_id") != campaign_id:
                continue
        metrics = json.loads(metrics_path.read_text())
        manifest = {}
        if (run_dir / "manifest.json").exists():
            manifest = json.loads((run_dir / "manifest.json").read_text())
        components = {}
        if (run_dir / "component_metrics.json").exists():
            components = json.loads((run_dir / "component_metrics.json").read_text())
        history = {}
        if (run_dir / "history.json").exists():
            history = json.loads((run_dir / "history.json").read_text())
        name = manifest.get("experiment_name") or run_dir.name.removeprefix(prefix)
        records.append(
            {
                "name": name,
                "run_id": run_dir.name,
                "status": "OK",
                "dataset": dataset,
                "campaign_id": campaign_id,
                "family": manifest.get("family"),
                "seed": manifest.get("seed", metrics.get("seed")),
                "split_lock_id": manifest.get("split_lock_id"),
                "split_protocol": manifest.get("split_protocol"),
                "oov_graph_rate": manifest.get("oov_graph_rate"),
                "oov_line_rate": manifest.get("oov_line_rate"),
                "run_dir": str(run_dir),
                "test_f1": metrics.get("test_f1"),
                "test_pr_auc": metrics.get("test_pr_auc"),
                "test_roc_auc": metrics.get("test_roc_auc"),
                "val_f1": metrics.get("val_f1"),
                "val_pr_auc": metrics.get("val_pr_auc"),
                "val_roc_auc": metrics.get("val_roc_auc"),
                "test_precision": metrics.get("test_precision"),
                "test_recall": metrics.get("test_recall"),
                "best_threshold": metrics.get("best_threshold"),
                "component_metrics": components,
                "history": _history_from_epochs(history.get("epochs") or metrics.get("history")),
            }
        )

    if not records:
        raise FileNotFoundError(
            f"No completed runs found for campaign {campaign_id!r} under {output_root}"
        )

    baseline_name = resolve_campaign_baseline_name(
        records, dataset=dataset, baseline_name=baseline_name
    )
    frame = pd.DataFrame(records)
    for column in ("test_f1", "test_pr_auc", "test_roc_auc"):
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.sort_values("test_pr_auc", ascending=False, na_position="last")
    csv_path = campaign_dir / "leaderboard.csv"
    json_path = campaign_dir / "leaderboard.json"
    frame.drop(columns=["component_metrics", "history"], errors="ignore").to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(records, indent=2, default=str))

    summary_frame, paired_frame = _seeded_campaign_statistics(frame, baseline_name)
    summary_frame.to_csv(campaign_dir / "summary_by_arm.csv", index=False)
    paired_frame.to_csv(campaign_dir / "paired_bootstrap_ci.csv", index=False)
    time_block_frame = _bgl_time_block_bootstrap(records, baseline_name=baseline_name)
    if not time_block_frame.empty:
        time_block_frame.to_csv(campaign_dir / "paired_time_block_bootstrap.csv", index=False)

    delta_path = campaign_dir / "delta_vs_baseline.csv"
    if not paired_frame.empty:
        paired_frame.to_csv(delta_path, index=False)

    _write_campaign_figures(campaign_dir, records, baseline_name=baseline_name)
    readme = campaign_dir / "README.md"
    readme.write_text(_campaign_readme(campaign_id, dataset, frame, baseline_name))
    return json_path


def _bgl_time_block_bootstrap(
    records: list[dict[str, Any]], *, baseline_name: str, samples: int = 2_000
) -> pd.DataFrame:
    from sklearn.metrics import average_precision_score

    baseline_records = [item for item in records if item.get("name") == baseline_name]
    rows: list[dict[str, Any]] = []
    for base in baseline_records:
        for arm in records:
            if arm.get("name") == baseline_name or arm.get("seed") != base.get("seed"):
                continue
            base_path = Path(str(base["run_dir"])) / "scores" / "test_component_scores.csv"
            arm_path = Path(str(arm["run_dir"])) / "scores" / "test_component_scores.csv"
            if not base_path.exists() or not arm_path.exists():
                continue
            merged = pd.read_csv(base_path)[["graph_id", "label", "score"]].merge(
                pd.read_csv(arm_path)[["graph_id", "label", "score"]],
                on=["graph_id", "label"], suffixes=("_baseline", "_arm"),
            )
            if merged.empty or merged["label"].nunique() != 2:
                continue
            try:
                graph_ids = pd.to_numeric(merged["graph_id"], errors="raise").astype(np.int64)
            except (ValueError, TypeError):
                continue
            unix_start = graph_ids % BGL_SPLIT_STRIDE
            for days in (1, 7):
                blocks = (unix_start // (days * 86_400)).to_numpy()
                unique = np.unique(blocks)
                if len(unique) < 2:
                    continue
                rng = np.random.default_rng(20_260 + int(base["seed"]))
                deltas: list[float] = []
                for _ in range(samples):
                    picked = rng.choice(unique, size=len(unique), replace=True)
                    indices = np.concatenate([np.flatnonzero(blocks == item) for item in picked])
                    labels = merged["label"].to_numpy()[indices]
                    if len(np.unique(labels)) != 2:
                        continue
                    deltas.append(float(
                        average_precision_score(labels, merged["score_arm"].to_numpy()[indices])
                        - average_precision_score(labels, merged["score_baseline"].to_numpy()[indices])
                    ))
                if deltas:
                    rows.append({
                        "baseline": baseline_name, "name": arm["name"], "seed": arm["seed"],
                        "block_days": days, "n_blocks": len(unique), "n_bootstrap": len(deltas),
                        "ap_delta": float(average_precision_score(merged["label"], merged["score_arm"])
                                          - average_precision_score(merged["label"], merged["score_baseline"])),
                        "ci95_low": float(np.quantile(deltas, 0.025)),
                        "ci95_high": float(np.quantile(deltas, 0.975)),
                        "bootstrap_unit": "paired_time_block",
                    })
    return pd.DataFrame(rows)


def _seeded_campaign_statistics(
    frame: pd.DataFrame,
    baseline_name: str,
    *,
    bootstrap_samples: int = 10_000,
    bootstrap_seed: int = 2026,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics = ("test_f1", "test_pr_auc", "test_roc_auc")
    summary_rows: list[dict[str, Any]] = []
    for name, group in frame.groupby("name", sort=True):
        row: dict[str, Any] = {"name": name, "n_seeds": int(group["seed"].nunique())}
        for metric in metrics:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            row[f"{metric}_mean"] = float(values.mean()) if len(values) else np.nan
            row[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        summary_rows.append(row)

    paired_rows: list[dict[str, Any]] = []
    baseline = frame[frame["name"] == baseline_name]
    if not baseline.empty and baseline["seed"].notna().all():
        rng = np.random.default_rng(bootstrap_seed)
        pair_keys = ["seed"]
        if "split_lock_id" in frame.columns and baseline["split_lock_id"].notna().all():
            pair_keys.append("split_lock_id")
        for name, group in frame.groupby("name", sort=True):
            merged = baseline[[*pair_keys, *metrics]].merge(
                group[[*pair_keys, *metrics]], on=pair_keys, suffixes=("_baseline", "_arm")
            )
            for metric in metrics:
                delta = (
                    pd.to_numeric(merged[f"{metric}_arm"], errors="coerce")
                    - pd.to_numeric(merged[f"{metric}_baseline"], errors="coerce")
                ).dropna().to_numpy(dtype=float)
                if not len(delta):
                    continue
                sampled = rng.choice(delta, size=(bootstrap_samples, len(delta)), replace=True).mean(axis=1)
                paired_rows.append(
                    {
                        "name": name,
                        "metric": metric,
                        "n_paired_seeds": int(len(delta)),
                        "mean_delta": float(delta.mean()),
                        "ci95_low": float(np.quantile(sampled, 0.025)),
                        "ci95_high": float(np.quantile(sampled, 0.975)),
                        "bootstrap_unit": "training_seed",
                        "pairing_keys": "+".join(pair_keys),
                    }
                )
    return pd.DataFrame(summary_rows), pd.DataFrame(paired_rows)


def _history_from_epochs(history: Any) -> dict[str, list[float]]:
    if isinstance(history, dict) and "total" in history:
        return history
    if isinstance(history, list):
        return {
            "total": [float(entry.get("train_total_loss", 0.0)) for entry in history],
            "structure": [float(entry.get("train_structure_loss", 0.0)) for entry in history],
            "node": [float(entry.get("train_node_loss", 0.0)) for entry in history],
            "edge": [float(entry.get("train_edge_loss", 0.0)) for entry in history],
        }
    return {}


def _write_campaign_figures(
    campaign_dir: Path,
    records: list[dict[str, Any]],
    *,
    baseline_name: str,
) -> None:
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import matplotlib.ticker as mticker
    except ImportError:
        return

    ok = [record for record in records if record.get("status") == "OK"]
    if not ok:
        return
    names = [record["name"] for record in ok]
    x = np.arange(len(names))
    figure, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, metric, label in zip(
        axes,
        ("test_f1", "test_pr_auc", "test_roc_auc"),
        ("Test F1", "Test PR-AUC", "Test ROC-AUC"),
    ):
        values = [float(record.get(metric) or 0.0) for record in ok]
        bars = axis.barh(x, values, color=plt.cm.tab10.colors[: len(names)], edgecolor="white", height=0.6)
        if values:
            bars[int(np.argmax(values))].set_edgecolor("#111")
            bars[int(np.argmax(values))].set_linewidth(2)
        for bar, value in zip(bars, values):
            axis.text(value + 0.002, bar.get_y() + bar.get_height() / 2, f"{value:.4f}", va="center", fontsize=8)
        axis.set_yticks(x)
        axis.set_yticklabels(names, fontsize=9)
        axis.set_xlabel(label)
        axis.invert_yaxis()
        axis.grid(axis="x", alpha=0.3)
        axis.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    figure.suptitle("Ablation comparison", fontsize=14, fontweight="bold")
    figure.tight_layout()
    figure.savefig(campaign_dir / "ablation_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

    histories = {record["name"]: record.get("history") or {} for record in ok if record.get("history")}
    if histories:
        n_exp = len(histories)
        figure, axes = plt.subplots(1, n_exp, figsize=(5 * n_exp, 4), squeeze=False)
        for axis, (name, hist) in zip(axes[0], histories.items()):
            epochs = range(1, len(hist.get("total") or []) + 1)
            axis.plot(epochs, hist.get("total") or [], "ko-", lw=2, ms=4, label="Total")
            axis.plot(epochs, hist.get("structure") or [], "bo--", lw=1.4, ms=3, label="Structure")
            axis.plot(epochs, hist.get("node") or [], "go-.", lw=1.4, ms=3, label="Node")
            axis.plot(epochs, hist.get("edge") or [], "ro:", lw=1.4, ms=3, label="Edge")
            axis.set_title(name, fontsize=9)
            axis.set_xlabel("Epoch")
            axis.set_ylabel("Loss")
            axis.legend(fontsize=7)
            axis.grid(alpha=0.3)
        figure.suptitle("Training loss curves", fontsize=13, fontweight="bold")
        figure.tight_layout()
        figure.savefig(campaign_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
        plt.close(figure)

    component_rows = []
    for record in ok:
        combined = (record.get("component_metrics") or {}).get("combined") or {}
        node = (record.get("component_metrics") or {}).get("node") or {}
        edge = (record.get("component_metrics") or {}).get("edge") or {}
        structure = (record.get("component_metrics") or {}).get("structure") or {}
        if combined or node:
            component_rows.append(
                {
                    "name": record["name"],
                    "combined": combined.get("roc_auc", 0.0),
                    "structure": structure.get("roc_auc", 0.0),
                    "node": node.get("roc_auc", 0.0),
                    "edge": edge.get("roc_auc", 0.0),
                }
            )
    if component_rows:
        figure, axis = plt.subplots(figsize=(10, 5))
        index = np.arange(len(component_rows))
        width = 0.2
        for offset, key, color in (
            (-1.5, "combined", "black"),
            (-0.5, "structure", "#4C72B0"),
            (0.5, "node", "#55A868"),
            (1.5, "edge", "#C44E52"),
        ):
            axis.bar(index + offset * width, [row[key] for row in component_rows], width, label=key, color=color)
        axis.set_xticks(index)
        axis.set_xticklabels([row["name"] for row in component_rows], rotation=30, ha="right")
        axis.set_ylabel("ROC-AUC")
        axis.set_ylim(0, 1.05)
        axis.legend()
        axis.set_title("Component ROC-AUC vs combined")
        axis.grid(axis="y", alpha=0.3)
        figure.tight_layout()
        figure.savefig(campaign_dir / "component_comparison.png", dpi=150, bbox_inches="tight")
        plt.close(figure)


def _campaign_readme(
    campaign_id: str,
    dataset: str,
    frame: pd.DataFrame,
    baseline_name: str,
) -> str:
    generated = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    lines = [
        f"# {dataset.upper()} ablation campaign `{campaign_id}`",
        "",
        f"Generated {generated}. Primary ranking metric: **test PR-AUC**.",
        "",
        "PR-AUC is threshold-free; chance is the positive-class rate, not 0.5. "
        "Test F1 uses the validation-F1 threshold (one operating point). "
        "ROC-AUC is secondary (chance = 0.5) and can look strong under class imbalance.",
        "",
        "| name | test_f1 | test_pr_auc | test_roc_auc | val_f1 |",
        "|---|---:|---:|---:|---:|",
    ]
    for _, row in frame.iterrows():
        marker = " **(baseline)**" if row["name"] == baseline_name else ""
        lines.append(
            f"| {row['name']}{marker} | {row.get('test_f1'):.4f} | {row.get('test_pr_auc'):.4f} "
            f"| {row.get('test_roc_auc'):.4f} | {row.get('val_f1'):.4f} |"
            if pd.notna(row.get("test_f1"))
            else f"| {row['name']}{marker} |  |  |  |  |"
        )
    lines.extend(
        [
            "",
            "Artifacts per run live under `outputs/{dataset}/{run_id}/` "
            "(metrics, figures, scores, checkpoint). This folder is the campaign rollup.",
            "",
        ]
    )
    return "\n".join(lines)


def resolved_training() -> dict[str, Any]:
    training = copy.deepcopy(TRAINING)
    if SMOKE:
        training["test_run"] = True
        training["epochs"] = 1
    return training


def assert_arm_meta(arm: str, meta: Mapping[str, Any] | None) -> None:
    global _LOCK_REF
    if not meta:
        raise FileNotFoundError(f"Missing dataset_meta.json for arm {arm!r}.")
    identity = meta.get("graph_identity") or {}
    split_lock = meta.get("split_lock_id")
    protocol = identity.get("split_protocol") or meta.get("split_protocol")
    fit_on = identity.get("fit_on") or meta.get("fit_on")
    n_train = int(meta.get("n_train") or 0)
    n_val = int(meta.get("n_val") or 0)
    n_test = int(meta.get("n_test") or 0)
    errors = []
    if protocol != "stratified":
        errors.append(f"split_protocol={protocol!r} (expected 'stratified')")
    if fit_on != "all":
        errors.append(f"fit_on={fit_on!r} (expected 'all')")
    if EXPECTED_SPLIT_LOCK_ID and split_lock != EXPECTED_SPLIT_LOCK_ID:
        errors.append(f"split_lock_id={split_lock!r} (expected {EXPECTED_SPLIT_LOCK_ID!r})")
    marker = (split_lock, n_train, n_val, n_test)
    if _LOCK_REF is None:
        _LOCK_REF = marker
    elif marker != _LOCK_REF:
        errors.append(f"split counts/lock {marker} disagree with first arm {_LOCK_REF}")
    if errors:
        raise ValueError(f"Arm {arm!r} is not compatible with this notebook: " + "; ".join(errors))


def stage_arm_graph(arm: str) -> Path:
    campaign_dir = Path(CAMPAIGN_DIR)
    source_gz = campaign_dir / "graphs" / arm / "graph_dataset.pt.gz"
    source_pt = campaign_dir / "graphs" / arm / "graph_dataset.pt"
    source_meta = campaign_dir / "graphs" / arm / "dataset_meta.json"
    if not source_gz.exists() and not source_pt.exists():
        raise FileNotFoundError(
            f"Prepared graph for {arm!r} not found under {campaign_dir / 'graphs' / arm}. "
            f"Upload campaigns/{CAMPAIGN_ID} to Drive or run prepare cells first."
        )
    dest = (
        Path(WORKSPACE_ROOT)
        / "data"
        / "processed"
        / DATASET
        / f"{_safe_name(CAMPAIGN_ID)}_{_safe_name(arm)}_graph_dataset.pt"
    )
    dest.parent.mkdir(parents=True, exist_ok=True)
    if source_gz.exists():
        if not dest.exists() or dest.stat().st_mtime < source_gz.stat().st_mtime:
            gunzip_file(source_gz, dest)
        _copy_graph_splits(source_gz, dest)
    else:
        if not dest.exists() or dest.stat().st_mtime < source_pt.stat().st_mtime:
            shutil.copy2(source_pt, dest)
        _copy_graph_splits(source_pt, dest)
    if source_meta.exists():
        shutil.copy2(source_meta, dest.with_name("dataset_meta.json"))
    meta = load_bundle_meta(dest)
    assert_arm_meta(arm, meta)
    return dest


def _experiment_config(
    arm: str,
    seed: int,
    family: str = "train",
    *,
    training: dict[str, Any] | None = None,
    arch: dict[str, Any] | None = None,
) -> dict[str, Any]:
    training = training or resolved_training()
    arch = dict(arch or GAE_ARCH)
    return {
        "experiment": {
            "name": arm,
            "dataset": DATASET,
            "seed": seed,
            "campaign_id": CAMPAIGN_ID,
            "family": family,
            "run_id": f"{_safe_name(CAMPAIGN_ID)}_{_safe_name(arm)}_seed{seed}",
        },
        "training": training,
        "ablation": {
            "graph": {
                "gine_aggregation": arch["gine_aggregation"],
                "node_transformation": arch["node_transformation"],
                "structure_decoder": arch["structure_decoder"],
            },
            "fusion": {
                "node_reconstruct": arch["node_reconstruct"],
                "mode": arch["fusion_mode"],
                "modality_projection_dim": arch["modality_projection_dim"],
                "node_loss": arch["node_loss_mode"],
                "node_block_weights": arch["node_block_weights"],
            },
        },
    }


def train_graph_bundle(
    graph_path: Path,
    *,
    training: dict[str, Any],
    seed: int,
    bundle_meta: dict[str, Any] | None = None,
    arch: dict[str, Any] | None = None,
) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any]]:
    from sklearn.metrics import confusion_matrix, precision_score, recall_score
    from torch.optim import Adam
    from torch_geometric.loader import DataLoader

    seed_everything(seed)
    arch = dict(arch or GAE_ARCH)
    device = get_device()
    train_graphs, val_graphs, test_graphs, split_meta = load_graph_splits(graph_path)
    meta = flatten_split_meta(split_meta, bundle_meta)
    node_dim = int(meta.get("node_dim") or split_meta.get("node_dim") or 0)
    edge_dim = int(meta.get("edge_dim") or split_meta.get("edge_dim") or 0)
    if not node_dim or not edge_dim:
        bundle = torch.load(graph_path, weights_only=False, map_location="cpu")
        node_dim = int(bundle["node_dim"])
        edge_dim = int(bundle["edge_dim"])
    if training["train_mode"] == "clean":
        train_graphs = [graph for graph in train_graphs if graph.y.item() == 0]
    if not train_graphs:
        raise ValueError("No training graphs remain after applying train_mode.")

    if training["test_run"]:
        sample_count = int(training["test_samples"])
        train_graphs = train_graphs[:sample_count]
        val_graphs = val_graphs[:sample_count]
        test_graphs = test_graphs[:sample_count]

    edge_mean, edge_std = _normalise_edge_attributes(
        train_graphs,
        val_graphs,
        test_graphs,
        enabled=bool(training["pre_normalize_edges"]),
        minimum_std=float(training["minimum_edge_std"]),
    )
    batch_size = int(training["batch_size"])
    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=batch_size)
    test_loader = DataLoader(test_graphs, batch_size=batch_size)

    model = AttributeAwareGAE(
        node_dim=int(node_dim),
        edge_dim=int(edge_dim),
        hidden_dim=int(training["hidden_dim"]),
        latent_dim=int(training["latent_dim"]),
        gine_aggregation=str(arch["gine_aggregation"]),
        node_transformation=str(arch["node_transformation"]),
        structure_decoder=str(arch["structure_decoder"]),
        node_recon_index=_node_recon_index_for_bundle(
            node_dim=int(node_dim),
            node_reconstruct=str(arch["node_reconstruct"]),
            split_meta=split_meta,
            bundle_meta=bundle_meta,
        ),
        tfidf_dim=int(meta.get("tfidf_dim") or 0),
        sbert_dim=int(meta.get("sbert_dim") or 0),
        fusion_mode=str(arch["fusion_mode"]),
        modality_projection_dim=int(arch["modality_projection_dim"]),
        node_loss_mode=str(arch["node_loss_mode"]),
        node_block_weights=dict(arch["node_block_weights"]),
    ).to(device)
    optimizer = Adam(model.parameters(), lr=float(training["learning_rate"]))
    loss_args = {
        "alpha": float(training["alpha"]),
        "beta": float(training["beta"]),
        "gamma": float(training["gamma"]),
    }
    history: list[dict[str, float | int]] = []
    best_state: dict[str, Any] | None = None
    best_val_f1 = -1.0
    best_threshold = 0.5

    for epoch in range(1, int(training["epochs"]) + 1):
        total, structure, node, edge = train_epoch(
            model, train_loader, optimizer, device, **loss_args
        )
        val_scores, val_labels = compute_anomaly_scores(model, val_loader, device, **loss_args)
        threshold, val_metrics = _threshold_and_metrics(val_labels, val_scores)
        history.append(
            {
                "epoch": epoch,
                "train_total_loss": total,
                "train_structure_loss": structure,
                "train_node_loss": node,
                "train_edge_loss": edge,
                "val_f1": val_metrics["f1"],
                "val_pr_auc": val_metrics["pr_auc"],
                "val_roc_auc": val_metrics["roc_auc"],
            }
        )
        print(
            f"epoch {epoch:02d}/{int(training['epochs']):02d}  "
            f"loss={total:.4f}  val_f1={val_metrics['f1']:.4f}  "
            f"val_pr_auc={val_metrics['pr_auc']:.4f}  device={device}"
        )
        if val_metrics["f1"] >= best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_threshold = threshold
            best_state = copy.deepcopy(model.state_dict())

    if best_state is None:
        raise RuntimeError("Training did not produce a validation checkpoint.")
    model.load_state_dict(best_state)
    val_scores, val_labels = compute_anomaly_scores(model, val_loader, device, **loss_args)
    _, val_metrics = _threshold_and_metrics(val_labels, val_scores, threshold=best_threshold)
    test_scores, test_labels, test_components, test_ids = compute_anomaly_scores(
        model, test_loader, device, return_components=True, **loss_args
    )
    test_metrics = _score_metrics(test_labels, test_scores, best_threshold)
    test_predictions = (test_scores > best_threshold).astype(int)
    matrix = confusion_matrix(test_labels, test_predictions, labels=[0, 1]).tolist()
    test_metrics.update(
        {
            "precision": _rounded(precision_score(test_labels, test_predictions, zero_division=0)),
            "recall": _rounded(recall_score(test_labels, test_predictions, zero_division=0)),
            "confusion_matrix": matrix,
        }
    )
    metrics: dict[str, Any] = {
        "best_threshold": _rounded(best_threshold),
        "val_f1": val_metrics["f1"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_roc_auc": val_metrics["roc_auc"],
        "test_f1": test_metrics["f1"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_confusion_matrix": test_metrics["confusion_matrix"],
        "device": str(device),
        "n_train": len(train_graphs),
        "n_val": len(val_graphs),
        "n_test": len(test_graphs),
        "history": history,
        "history_epochs": history,
    }
    checkpoint = {
        "model_state_dict": best_state,
        "node_dim": int(node_dim),
        "edge_dim": int(edge_dim),
        "hidden_dim": int(training["hidden_dim"]),
        "latent_dim": int(training["latent_dim"]),
        "best_threshold": best_threshold,
        "training": training,
        "history": _loss_history(history),
        "edge_mean": edge_mean,
        "edge_std": edge_std,
        "gine_aggregation": arch["gine_aggregation"],
        "node_transformation": arch["node_transformation"],
        "structure_decoder": arch["structure_decoder"],
        "node_reconstruct": arch["node_reconstruct"],
        "node_recon_dim": int(model.node_recon_index.numel()),
        "node_recon_index": model.node_recon_index.detach().cpu().tolist(),
        "tfidf_dim": model.tfidf_dim,
        "sbert_dim": model.sbert_dim,
        "fusion_mode": arch["fusion_mode"],
        "modality_projection_dim": arch["modality_projection_dim"],
        "node_loss_mode": arch["node_loss_mode"],
        "node_block_weights": arch["node_block_weights"],
    }
    metrics["history"] = _loss_history(history)
    eval_payload = {
        "test_scores": test_scores,
        "test_labels": test_labels,
        "structure": test_components["structure"],
        "node": test_components["node"],
        "edge": test_components["edge"],
        "tfidf": test_components["tfidf"],
        "sbert": test_components["sbert"],
        "extras": test_components["extras"],
        "graph_ids": test_ids,
    }
    return metrics, checkpoint, eval_payload


def train_isolation_forest_bundle(graph_path: Path, *, seed: int) -> tuple[dict[str, Any], dict[str, Any]]:
    from sklearn.ensemble import IsolationForest
    from sklearn.metrics import confusion_matrix, precision_score, recall_score

    train_graphs, val_graphs, test_graphs, _ = load_graph_splits(graph_path)
    clean_train = [graph for graph in train_graphs if int(graph.y.item()) == 0]
    if not clean_train:
        raise ValueError("No normal training graphs remain for Isolation Forest.")
    x_train, x_val, x_test = window_feature_matrices(clean_train, val_graphs, test_graphs)
    model = IsolationForest(
        n_estimators=300,
        max_samples=min(256, len(x_train)),
        contamination="auto",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(x_train)
    val_scores = -model.score_samples(x_val)
    val_labels = np.asarray([int(graph.y.item()) for graph in val_graphs])
    threshold, val_metrics = _threshold_and_metrics(val_labels, val_scores)
    test_scores = -model.score_samples(x_test)
    test_labels = np.asarray([int(graph.y.item()) for graph in test_graphs])
    test_metrics = _score_metrics(test_labels, test_scores, threshold)
    predictions = (test_scores > threshold).astype(int)
    test_metrics.update(
        {
            "precision": _rounded(precision_score(test_labels, predictions, zero_division=0)),
            "recall": _rounded(recall_score(test_labels, predictions, zero_division=0)),
            "confusion_matrix": confusion_matrix(test_labels, predictions, labels=[0, 1]).tolist(),
        }
    )
    metrics = {
        "best_threshold": _rounded(threshold),
        "val_f1": val_metrics["f1"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_roc_auc": val_metrics["roc_auc"],
        "test_f1": test_metrics["f1"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_confusion_matrix": test_metrics["confusion_matrix"],
        "n_train": len(clean_train),
        "n_val": len(val_graphs),
        "n_test": len(test_graphs),
        "baseline": "isolation_forest",
        "n_estimators": 300,
        "max_samples": min(256, len(x_train)),
    }
    payload = {
        "test_scores": test_scores,
        "test_labels": test_labels,
        "graph_ids": sequence_ids_from_graphs(test_graphs),
    }
    return metrics, payload


def run_gae_arm(arm: str) -> list[Path]:
    graph_path = Path(GRAPH_DATASET_PATH)
    bundle_meta = load_bundle_meta(graph_path) or {}
    training, arch = resolved_arm(arm)
    written: list[Path] = []
    for seed in SEEDS:
        run_id = f"{_safe_name(CAMPAIGN_ID)}_{_safe_name(arm)}_seed{seed}"
        output_dir = Path(WORKSPACE_ROOT) / "outputs" / DATASET / run_id
        if (output_dir / "metrics.json").exists():
            print(f"[SKIP] {arm} seed={seed} already completed at {output_dir}")
            push_run_outputs(output_dir)
            written.append(output_dir)
            continue
        print(f"[TRAIN] {arm} seed={seed} graph={graph_path}  overrides={ARM_OVERRIDES[arm]}")
        metrics, checkpoint, eval_payload = train_graph_bundle(
            graph_path, training=training, seed=seed, bundle_meta=bundle_meta, arch=arch
        )
        output_dir.mkdir(parents=True, exist_ok=True)
        torch.save(checkpoint, output_dir / "attribute_gae.pt")
        campaign_meta = {
            "campaign_id": CAMPAIGN_ID,
            "family": "train",
            "experiment_name": arm,
            "run_id": run_id,
            "graph_identity": bundle_meta.get("graph_identity"),
            "split_lock_id": bundle_meta.get("split_lock_id"),
            "parent_graph": str(graph_path),
            "feature_contract": bundle_meta.get("feature_contract"),
            "seed": seed,
            "split_protocol": (bundle_meta.get("graph_identity") or {}).get("split_protocol", "stratified"),
            "oov_graph_rate": bundle_meta.get("oov_graph_rate"),
            "oov_line_rate": bundle_meta.get("oov_line_rate"),
        }
        write_eval_pack(
            output_dir,
            metrics={key: value for key, value in metrics.items() if key != "history_epochs"},
            history_epochs=metrics["history_epochs"],
            labels=eval_payload["test_labels"],
            scores=eval_payload["test_scores"],
            structure=eval_payload["structure"],
            node=eval_payload["node"],
            edge=eval_payload["edge"],
            node_blocks={name: eval_payload[name] for name in ("tfidf", "sbert", "extras")},
            threshold=float(metrics["best_threshold"]),
            alpha=float(training["alpha"]),
            beta=float(training["beta"]),
            gamma=float(training["gamma"]),
            graph_ids=eval_payload["graph_ids"],
            config=_experiment_config(arm, seed, training=training, arch=arch),
            campaign_meta=campaign_meta,
        )
        figures = output_dir / "figures"
        print(
            f"[DONE] {arm} seed={seed}  test_pr_auc={metrics['test_pr_auc']}  "
            f"test_f1={metrics['test_f1']}  test_roc_auc={metrics['test_roc_auc']} → {output_dir}"
        )
        print(
            "[PLOTS]",
            figures / "confusion_matrix.png",
            figures / "training_history.png",
            figures / "test_pr_curve.png",
            figures / "test_score_distribution.png",
        )
        push_run_outputs(output_dir)
        written.append(output_dir)
    return written


## Family B arms

Each cell trains the **same** 20260818 HDFS graphs with one model override from `configs/ablation_train.yaml`. Skip completed `metrics.json`.


### `baseline_full`

BGL Family B baseline: GINE-sum, MLP node transform, directed MLP structure decoder, concat fusion, α=β=γ=1, lr=0.001, hidden=128, latent=64.


In [ ]:
ARM = "baseline_full"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `gine_mean_agg`

GINE aggregation `sum` → `mean`.


In [ ]:
ARM = "gine_mean_agg"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `gine_max_agg`

GINE aggregation `sum` → `max`.


In [ ]:
ARM = "gine_max_agg"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `linear_node_transform`

GINE node transform `mlp` → `linear`.


In [ ]:
ARM = "linear_node_transform"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `latent_32`

Latent width 64 → 32.


In [ ]:
ARM = "latent_32"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `hidden_256`

Hidden width 128 → 256.


In [ ]:
ARM = "hidden_256"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `learning_rate_001`

Adam lr stays 0.001 (same as this campaign default and the BGL baseline). Kept for matrix identity with `ablation_train.yaml`; it is a no-op vs `baseline_full`.


In [ ]:
ARM = "learning_rate_001"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `gamma_05`

Edge-reconstruction weight γ 1.0 → 0.5.


In [ ]:
ARM = "gamma_05"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `inner_product_structure`

Structure decoder `mlp` (directed) → `inner_product` (symmetric).


In [ ]:
ARM = "inner_product_structure"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `alpha_0`

Drop structure loss (α=0).


In [ ]:
ARM = "alpha_0"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `beta_0`

Drop node-feature reconstruction (β=0).


In [ ]:
ARM = "beta_0"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `gamma_0`

Drop edge-attribute reconstruction (γ=0).


In [ ]:
ARM = "gamma_0"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `lexical_node_recon`

Node decoder reconstructs TF-IDF + extras only (`without_sbert`).


In [ ]:
ARM = "lexical_node_recon"
print(f"Arm: {ARM}  train={RUN_TRAIN} smoke={SMOKE} overrides={ARM_OVERRIDES[ARM]}")
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


## Campaign comparison

Rank by test PR-AUC vs `baseline_full`. Chance is the HDFS positive-class rate (about 0.03). Paired seed CIs are the uncertainty estimate on this split.


In [ ]:
import pandas as pd
from IPython.display import display

report_path = write_campaign_report(
    WORKSPACE_ROOT,
    dataset="hdfs",
    campaign_id=_safe_name(CAMPAIGN_ID),
    baseline_name="baseline_full",
)
campaign_out = Path(WORKSPACE_ROOT) / "outputs" / "hdfs" / "campaigns" / _safe_name(CAMPAIGN_ID)
print(f"Campaign report: {report_path}")

leaderboard = pd.read_csv(campaign_out / "leaderboard.csv")
summary = pd.read_csv(campaign_out / "summary_by_arm.csv")
paired = pd.read_csv(campaign_out / "paired_bootstrap_ci.csv")
print("Leaderboard (all seeds, sorted by test PR-AUC):")
display(leaderboard[["name", "seed", "test_pr_auc", "test_f1", "test_roc_auc", "val_f1"]])
print("Means ± std by arm:")
display(summary)
print("Paired seed-bootstrap deltas vs baseline_full:")
display(paired)

readme = campaign_out / "README.md"
if readme.exists():
    print(readme.read_text())
push_to_drive(campaign_out)


## Optional: copy leftover outputs to Drive (Colab)

Each seed is already pushed when it finishes. Run this only as a catch-all.


In [ ]:
if IN_COLAB:
    source = WORKSPACE_ROOT / "outputs"
    destination = DRIVE_ARTIFACT_ROOT / "outputs"
    if source.exists():
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print(f"Copied {source} → {destination}")
    else:
        print("No outputs/ to copy.")
else:
    print(f"Local outputs: {Path(WORKSPACE_ROOT) / 'outputs' / 'hdfs'}")
